In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('new_synthetic_agri_data_india_fixed.csv')
print(df.shape)
df.head()

(5000, 23)


,Year,Region,Season,Soil Type,Soil pH,Soil Nitrogen,Soil Phosphorus,Soil Potassium,Soil Organic Matter (%),Soil Moisture (%),Avg Rainfall (mm),Solar Radiation Impact (BTU/sqft),Rotation Sequence,Crop_Planted (Action),Growth Duration (days),Water Requirement (mm),Phosphorus Requirement (kg/ha),Potassium Requirement (kg/ha),Nitrogen Requirement (kg/ha),Base Yield (kg/ha),Simulated Yield (kg/ha),Soil Score Before,Soil Score After
0,2006,Haryana,Zaid,Alluvial,6.60,119.83,23.67,73.39,3.79,19.46,700,24,"['Rice', 'Pigeon Pea', 'Tomato']",Tomato,80,350,20,30,70,10000,11834.35,0.86,0.88
1,2010,Arunachal Pradesh,Kharif,Laterite,5.83,113.31,19.02,51.24,1.04,14.70,2500,15,"['Groundnut', 'Pigeon Pea', 'Cucumber']",Groundnut,90,400,20,30,50,1200,1141.00,0.64,0.64
2,2005,Manipur,Rabi,Laterite,5.02,70.04,24.88,48.52,2.22,5.07,1800,17,"['Rice', 'Chickpea', 'Tomato']",Chickpea,90,350,20,25,50,1000,1244.16,0.54,0.54
3,2009,Meghalaya,Zaid,Red,5.88,62.33,29.61,46.98,1.18,16.18,2500,15,"['Groundnut', 'Chickpea', 'Tomato']",Tomato,80,350,20,30,70,10000,9795.23,0.62,0.63
4,2019,Chhattisgarh,Kharif,Red,6.45,89.62,15.70,58.23,1.34,10.65,1300,25,"['Rice', 'Pigeon Pea', 'Chili']",Rice,140,1500,40,60,100,4000,1569.68,0.60,0.60


In [ ]:
def extract_previous_crop(row):
    seq = ast.literal_eval(row['Rotation Sequence'])
    crop = row['Crop_Planted (Action)']
    idx = seq.index(crop)
    return seq[idx - 1] if idx > 0 else 'None'

df['previous_crop'] = df.apply(extract_previous_crop, axis=1)
df['previous_crop'].value_counts()

previous_crop
None          1622
Chickpea       882
Pigeon Pea     867
Groundnut      419
Soybean        416
Rice           415
Maize          379
Name: count, dtype: int64

In [ ]:
print(df['Crop_Planted (Action)'].value_counts())
print()
print(df['Simulated Yield (kg/ha)'].describe())
print()
print("Regions:", df['Region'].nunique(), "| Seasons:", df['Season'].unique(), "| Soil types:", df['Soil Type'].nunique())

Crop_Planted (Action)
Pigeon Pea    836
Chickpea      793
Tomato        452
Groundnut     443
Soybean       436
Chili         433
Cucumber      433
Okra          431
Rice          389
Maize         354
Name: count, dtype: int64

count     5000.000000
mean      3144.281554
std       3222.679003
min        223.000000
25%       1279.055000
50%       1950.280000
75%       3428.002500
max      21484.830000
Name: Simulated Yield (kg/ha), dtype: float64

Regions: 37 | Seasons: <StringArray>
['Zaid', 'Kharif', 'Rabi']
Length: 3, dtype: str | Soil types: 11


In [ ]:
req_cols = ['Growth Duration (days)', 'Water Requirement (mm)',
            'Phosphorus Requirement (kg/ha)', 'Potassium Requirement (kg/ha)',
            'Nitrogen Requirement (kg/ha)', 'Base Yield (kg/ha)']

crop_requirements = df.groupby('Crop_Planted (Action)')[req_cols].first()
crop_requirements

,Growth Duration (days),Water Requirement (mm),Phosphorus Requirement (kg/ha),Potassium Requirement (kg/ha),Nitrogen Requirement (kg/ha),Base Yield (kg/ha)
Crop_Planted (Action),,,,,,
Chickpea,90,350,20,25,50,1000
Chili,70,300,15,20,60,1500
Cucumber,70,250,10,20,40,3000
Groundnut,90,400,20,30,50,1200
Maize,110,600,35,50,80,6000
Okra,60,250,15,20,50,2000
Pigeon Pea,120,600,30,40,50,1800
Rice,140,1500,40,60,100,4000
Soybean,100,500,25,35,50,2000


In [ ]:
print("corr(Base Yield, Simulated Yield):", df['Base Yield (kg/ha)'].corr(df['Simulated Yield (kg/ha)']))

corr(Base Yield, Simulated Yield): 0.8010580675425589


In [ ]:
soil_cols = ['Soil pH','Soil Nitrogen','Soil Phosphorus','Soil Potassium',
             'Soil Organic Matter (%)','Soil Moisture (%)','Soil Score Before']
df['yield_ratio'] = df['Simulated Yield (kg/ha)'] / df['Base Yield (kg/ha)']
print(df[soil_cols + ['yield_ratio']].corr()['yield_ratio'].sort_values())

Soil pH                    0.241678
Soil Organic Matter (%)    0.344244
Soil Nitrogen              0.390361
Soil Potassium             0.451612
Soil Moisture (%)          0.457233
Soil Score Before          0.558260
Soil Phosphorus            0.631677
yield_ratio                1.000000
Name: yield_ratio, dtype: float64


In [ ]:
feature_cols = [
    'Region', 'Season', 'Soil Type', 'Soil pH', 'Soil Nitrogen', 'Soil Phosphorus',
    'Soil Potassium', 'Soil Organic Matter (%)', 'Soil Moisture (%)', 'Avg Rainfall (mm)',
    'Solar Radiation Impact (BTU/sqft)', 'Crop_Planted (Action)'
]
target_col = 'yield_ratio'

X = df[feature_cols]
y = df[target_col]

categorical_cols = ['Region', 'Season', 'Soil Type', 'Crop_Planted (Action)']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
], remainder='passthrough')

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("R2 (yield ratio):", r2_score(y_test, y_pred))
print("MAE (yield ratio):", mean_absolute_error(y_test, y_pred))

R2 (yield ratio): 0.9873400169304827
MAE (yield ratio): 0.021792496123148094


In [ ]:
ohe = pipeline.named_steps['preprocess'].named_transformers_['cat']
cat_names = ohe.get_feature_names_out(categorical_cols)
num_names = [c for c in feature_cols if c not in categorical_cols]
all_names = list(cat_names) + num_names

importances = pd.Series(pipeline.named_steps['model'].feature_importances_, index=all_names)
importances.sort_values(ascending=False).head(15)

Soil Phosphorus                      0.421809
Season_Zaid                          0.229355
Crop_Planted (Action)_Cucumber       0.141746
Crop_Planted (Action)_Rice           0.035801
Crop_Planted (Action)_Chickpea       0.032863
Crop_Planted (Action)_Groundnut      0.029414
Crop_Planted (Action)_Tomato         0.025279
Soil Nitrogen                        0.023982
Avg Rainfall (mm)                    0.013509
Crop_Planted (Action)_Soybean        0.008363
Soil Potassium                       0.007843
Solar Radiation Impact (BTU/sqft)    0.005996
Region_Ladakh                        0.005032
Crop_Planted (Action)_Pigeon Pea     0.004875
Crop_Planted (Action)_Maize          0.004534
dtype: float64

In [ ]:
ALL_CROPS = df['Crop_Planted (Action)'].unique().tolist()
base_yield_lookup = crop_requirements['Base Yield (kg/ha)']

def recommend_next_crop(field_state, previous_crop='None', top_n=3):
    """
    field_state: dict with keys Region, Season, Soil Type, Soil pH, Soil Nitrogen,
                 Soil Phosphorus, Soil Potassium, Soil Organic Matter (%), Soil Moisture (%),
                 Avg Rainfall (mm), Solar Radiation Impact (BTU/sqft)
    previous_crop: crop name string or 'None' -- shown in output only, not used by the model
                   (the soil reading in field_state is assumed to already reflect its effect)
    Returns a DataFrame of top_n candidate crops ranked by predicted yield (kg/ha).
    """
    rows = []
    for crop in ALL_CROPS:
        row = field_state.copy()
        row['Crop_Planted (Action)'] = crop
        rows.append(row)

    candidates = pd.DataFrame(rows)[feature_cols]
    ratio_pred = pipeline.predict(candidates)
    base = base_yield_lookup.loc[candidates['Crop_Planted (Action)']].values
    candidates['predicted_yield'] = ratio_pred * base
    candidates['recommended_after'] = previous_crop

    return candidates[['Crop_Planted (Action)', 'recommended_after', 'predicted_yield']]\
        .sort_values('predicted_yield', ascending=False)\
        .head(top_n)\
        .rename(columns={'Crop_Planted (Action)': 'crop'})\
        .reset_index(drop=True)

In [ ]:
example_field = {
    'Region': 'Haryana',
    'Season': 'Kharif',
    'Soil Type': 'Alluvial',
    'Soil pH': 6.6,
    'Soil Nitrogen': 120,
    'Soil Phosphorus': 35,
    'Soil Potassium': 73,
    'Soil Organic Matter (%)': 3.8,
    'Soil Moisture (%)': 20,
    'Avg Rainfall (mm)': 700,
    'Solar Radiation Impact (BTU/sqft)': 24
}

recommend_next_crop(example_field, previous_crop='Pigeon Pea', top_n=4)

,crop,recommended_after,predicted_yield
0,Tomato,Pigeon Pea,12472.428981
1,Maize,Pigeon Pea,6016.211433
2,Cucumber,Pigeon Pea,4990.197217
3,Soybean,Pigeon Pea,2787.378633


In [ ]:
low_p_field = example_field.copy()
low_p_field['Soil Phosphorus'] = 15
recommend_next_crop(low_p_field, previous_crop='Pigeon Pea', top_n=4)

,crop,recommended_after,predicted_yield
0,Tomato,Pigeon Pea,4466.773870
1,Maize,Pigeon Pea,2529.788978
2,Cucumber,Pigeon Pea,2214.698950
3,Rice,Pigeon Pea,1491.296033


In [ ]:
sample = df.sample(1, random_state=7).iloc[0]
field_state = {
    'Region': sample['Region'], 'Season': sample['Season'], 'Soil Type': sample['Soil Type'],
    'Soil pH': sample['Soil pH'], 'Soil Nitrogen': sample['Soil Nitrogen'],
    'Soil Phosphorus': sample['Soil Phosphorus'], 'Soil Potassium': sample['Soil Potassium'],
    'Soil Organic Matter (%)': sample['Soil Organic Matter (%)'],
    'Soil Moisture (%)': sample['Soil Moisture (%)'], 'Avg Rainfall (mm)': sample['Avg Rainfall (mm)'],
    'Solar Radiation Impact (BTU/sqft)': sample['Solar Radiation Impact (BTU/sqft)']
}

print("Actual crop planted:", sample['Crop_Planted (Action)'])
print("Actual previous crop:", sample['previous_crop'])
print("Actual simulated yield:", sample['Simulated Yield (kg/ha)'])
print()
print("Model recommendation:")
recommend_next_crop(field_state, previous_crop=sample['previous_crop'], top_n=4)

Actual crop planted: Rice
Actual previous crop: None
Actual simulated yield: 1728.2

Model recommendation:


,crop,recommended_after,predicted_yield
0,Tomato,None,4972.152426
1,Maize,None,2943.948689
2,Cucumber,None,2472.257206
3,Rice,None,1726.664067


In [ ]:
joblib.dump(pipeline, 'crop_rotation_yield_model.pkl')
crop_requirements.to_csv('crop_requirements.csv')

print("Saved: crop_rotation_yield_model.pkl, crop_requirements.csv")

Saved: crop_rotation_yield_model.pkl, crop_requirements.csv
